# 02 单比特 Lindblad + T1/T2 + 1/f 噪声（2 月）

目标：验证 T1/T2 与非静态噪声描述对单比特结果的影响。


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'examples' / 'backend.yaml').exists():
            return p
    raise FileNotFoundError('???????????? pyproject.toml ? examples/backend.yaml?')

ROOT = find_project_root(Path.cwd())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from qsim.ui.notebook import run_workflow

BACKEND_PATH = ROOT / 'examples' / 'backend.yaml'
OUT_ROOT = ROOT / 'runs' / 'roadmap_2026H1'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print('ROOT =', ROOT)
print('BACKEND_PATH =', BACKEND_PATH)
print('OUT_ROOT =', OUT_ROOT)


def run_case(tag: str, qasm_text: str, hardware: dict | None = None, noise: dict | None = None, engine: str = 'qutip'):
    out_dir = OUT_ROOT / tag
    return run_workflow(
        qasm_text=qasm_text,
        backend_path=str(BACKEND_PATH),
        out_dir=str(out_dir),
        hardware=hardware or {},
        noise=noise or {},
        engine=engine,
        persist_artifacts=True,
        export_dxf=False,
    )


def get_metric(result: dict, key: str, default: float = np.nan) -> float:
    obs = result.get('analysis', {}).get('observables', {}).get('values', {})
    return float(obs.get(key, default))

In [ ]:
QASM_1Q = """
OPENQASM 3;
qubit[1] q;
bit[1] c;
h q[0];
rz(0.5) q[0];
x q[0];
measure q[0] -> c[0];
"""

In [ ]:
t1_list = [20.0, 40.0, 80.0, 160.0]
final_p1_t1 = []
for t1 in t1_list:
    r = run_case(f'02_t1_{int(t1)}', QASM_1Q, noise={'model': 'markovian_lindblad', 't1': t1, 't2': 0.8*t1})
    final_p1_t1.append(get_metric(r, 'final_p1'))

amps = [0.0, 0.01, 0.02, 0.04]
final_p1_1f = []
for amp in amps:
    r = run_case(f'02_1f_{amp:.3f}', QASM_1Q, noise={'model': 'one_over_f', 'one_over_f_amp': amp, 'one_over_f_fmin': 1e-3, 'one_over_f_fmax': 0.2})
    final_p1_1f.append(get_metric(r, 'final_p1'))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].plot(t1_list, final_p1_t1, marker='o')
ax[0].set_title('T1/T2 ? final_p1 ???')
ax[0].set_xlabel('T1')
ax[0].set_ylabel('final_p1')
ax[0].grid(alpha=0.3)

ax[1].plot(amps, final_p1_1f, marker='o', color='tab:orange')
ax[1].set_title('1/f ??? final_p1 ???')
ax[1].set_xlabel('one_over_f_amp')
ax[1].set_ylabel('final_p1')
ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()